# Embodied Motion Flow - Training Tutorial
Kaggle full run: AIST++ training, EMA/CFG showcase, and one downloadable ZIP.


In [ ]:
import os, sys, subprocess, glob, json, shutil
from pathlib import Path
REPO_URL = "https://github.com/Blackhand01/Humanoid-Motion-Diffusion"
WORKDIR = Path("/kaggle/working/Embodied-Motion-Diffusion")
PYTHON = sys.executable
def sh(cmd: list[str]) -> None:
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True)
print("Python:", PYTHON)
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
sh(["git", "clone", REPO_URL, str(WORKDIR)])
os.chdir(WORKDIR)
print("Repo ready:", Path.cwd())


In [ ]:
packages = "numpy PyYAML scikit-learn tqdm matplotlib pandas imageio imageio-ffmpeg librosa soundfile".split()
sh([PYTHON, "-m", "pip", "install", "-q", *packages])
sh([PYTHON, "-m", "pip", "install", "-q", "-e", "."])
DATASET_BASE = Path("/kaggle/input/datasets/stefanoroybisignano/aist-smpl-clean/aist-smpl-clean")
if not DATASET_BASE.exists():
    found = glob.glob("/kaggle/input/**/motions", recursive=True); DATASET_BASE = Path(found[0]).parent if found else DATASET_BASE
paths = {"AISTPP_ROOT": DATASET_BASE/"motions", "AISTPP_SPLIT_ROOT": DATASET_BASE/"splits"}
paths["AISTPP_IGNORE_LIST"] = DATASET_BASE/"ignore_list.txt"; paths["AISTPP_AUDIO_ROOT"] = DATASET_BASE/"audio"
paths["SHOWCASE_TRACK"] = Path("/kaggle/input/datasets/stefanoroybisignano/stardustdata/stardust.mp3")
for key, value in paths.items(): os.environ[key] = str(value)
print(json.dumps({k: f"{v} :: {Path(v).exists()}" for k, v in os.environ.items() if k in paths}, indent=2))
sh([PYTHON, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"])


In [ ]:
zip_path = "/kaggle/working/embodied_motion_flow_showcase.zip"
cmd = [PYTHON, "run_pipeline.py", "full", "--config", "configs/kaggle_prod.yaml"]
cmd += ["--fresh-start", "--zip-path", zip_path]
print("Command:", " ".join(cmd))
print("Expected: dense AIST++ clips, 180 epochs, EMA+CFG showcase")
print("Targets: TSI < 1.0, JLVR < 0.10, failures = 0")
print("Viral MP4 includes the conditioning audio segment")
print("Final archive:", zip_path)
sh(cmd)
print("Pipeline completed")
print("Download the ZIP from the final cell")
print("-")


In [ ]:
from IPython.display import FileLink, Video, display
zip_file = Path("/kaggle/working/embodied_motion_flow_showcase.zip")
out = Path("/kaggle/working/outputs/showcase")
metrics = Path("/kaggle/working/outputs/metrics/evaluation_metrics.json")
print(metrics.read_text() if metrics.exists() else "missing metrics")
print("ZIP:", zip_file, "exists=", zip_file.exists())
if zip_file.exists(): display(FileLink(str(zip_file)))
for path in sorted(out.glob("*")): print(path)
viral = out / "stardust_0046_0101_viral.mp4"
research = out / "stardust_0046_0101_research.mp4"
if viral.exists(): display(Video(str(viral), embed=True, width=480))
if research.exists(): display(Video(str(research), embed=True, width=720))
